In [1]:
import sys
sys.path.append('../src/')

In [17]:
from constants import INPT_VARS, EXTRA_VARS, OUT_VARS, GLOBAL_COMBINED_STATS
import hydra
from hydra.utils import instantiate
from pathlib import Path
import os
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import logging

from utils.data_utils import (
    get_wet_mask,
    get_train_test_ranges,
    gen_data_in_test,
    gen_data_out_test,
    data_CNN_Lateral,
    data_CNN_Dynamic,
    gen_data_025_lateral,
    gen_data_global_new,
)
from utils.eval_utils import (
    generate_model_rollout,
    compute_mean,
    compute_var,
    compute_corrs_area,
    compute_rmse,
    compute_corrs,
    compute_KE,
    compute_time_spec,
    compute_ACC,
    compute_nino34,
    compute_amo,
    gen_KE_spectrum,
    gen_KE,
    gen_KE_range,
    gen_value_range,
    gen_enstrophy_spectrum,
    gen_enstrophy,
    compute_corrs_single,
    compute_ACC_single,
    compute_RMSE_single,
    compute_mean_single,
)
from utils.subgrid_utils import get_area_tensor
from utils.climate_utils import compute_laplacian_wet
from utils.plot_utils import (
    plot_short_time_stats,
    plot_long_time_stats,
    plot_map,
    plot_error_map,
    plot_both_error_map,
    plot_metrics_KE_spectrum,
    plot_metrics_KE,
    plot_metrics_enstrophy_spectrum,
    plot_metrics_entrophy,
    plot_metrics_corr,
    plot_metrics_rmse,
    plot_metrics_acc,
    plot_metrics_mean,
    plot_metrics_pdf,
    get_initial_snapshot_fig,
    plot_region_based_metric,
    plot_diff_map,
)

import numpy as np
import torch
import xarray as xr
import copy

In [18]:
class data_CNN_Dynamic(torch.utils.data.Dataset):

    def __init__(self, data_in, data_out, wet, std_dict=None, mode='default', device="cuda"):
        super().__init__()
        self.device = device
        num_inputs = data_in.shape[3]
        num_outputs = data_out.shape[3]
        self.size = data_in.shape[0]

        data_in = np.nan_to_num(data_in)
        data_out = np.nan_to_num(data_out)
        
        if std_dict is None:
            std_data = np.nanstd(data_in, axis=(0, 1, 2))
            mean_data = np.nanmean(data_in, axis=(0, 1, 2))
            std_label = np.nanstd(data_out, axis=(0, 1, 2))
            mean_label = np.nanmean(data_out, axis=(0, 1, 2))
        else:
            std_data = std_dict["s_in"]
            mean_data = std_dict["m_in"]
            std_label = std_dict["s_out"]
            mean_label = std_dict["m_out"]

        self.wet = wet

        if mode == 'minus':
            data_in[:, :, :, -1] = (data_in[:, :, :, -1] - 1)
        elif mode == 'plus':
            data_in[:, :, :, -1] = (data_in[:, :, :, -1] + 1)
        elif mode == 'default':
            pass
        else:
            raise NotImplementedError()
        # data_in[:, :, :, -1] = (data_in[:, :, :, -1] + 1)

        for i in range(num_inputs):
            data_in[:, :, :, i] = (data_in[:, :, :, i] - mean_data[i]) / std_data[i]

        for i in range(num_outputs):
            data_out[:, :, :, i] = (data_out[:, :, :, i] - mean_label[i]) / std_label[i]

        data_in = torch.from_numpy(data_in).type(torch.float32).to(device="cpu")
        data_out = torch.from_numpy(data_out).type(torch.float32).to(device="cpu")

        std_dict = {
            "s_in": std_data,
            "s_out": std_label,
            "m_in": mean_data,
            "m_out": mean_label,
        }

        if wet == None:
            self.input = torch.swapaxes(torch.swapaxes(data_in, 1, 3), 2, 3)
            self.output = torch.swapaxes(torch.swapaxes(data_out, 1, 3), 2, 3)

        else:
            self.input = torch.mul(
                torch.swapaxes(torch.swapaxes(data_in, 1, 3), 2, 3), wet
            )
            self.output = torch.mul(
                torch.swapaxes(torch.swapaxes(data_out, 1, 3), 2, 3), wet
            )

        self.norm_vals = std_dict

    def set_device(self, device):
        self.device = device

    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.size

    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        data_in = self.input[idx]
        label = self.output[idx]
        return data_in.to(device=self.device), label.to(device=self.device)


In [25]:

class Eval:
    def __init__(self, args, mode):
        # Getting input, extra input and output
        self.inputs = INPT_VARS[args.exp_num_in]
        self.extra_in = EXTRA_VARS[args.exp_num_extra]
        self.outputs = OUT_VARS[args.exp_num_out]

        self.str_in = "".join([i + "_" for i in self.inputs])
        self.str_ext = "".join([i + "_" for i in self.extra_in])
        self.str_out = "".join([i + "_" for i in self.outputs])

        print("inputs: " + self.str_in)
        print("extra inputs: " + self.str_ext)
        print("outputs: " + self.str_out)

        self.N_atm = len(self.extra_in)  # Number of atmosphere variables
        self.N_in = len(self.inputs)
        if args.lateral:
            self.N_extra = (
                self.N_atm + self.N_in
            )  # Number of atmosphere variables + Lateral boundary variables
        else:
            self.N_extra = self.N_atm  # Number of atmosphere variables
        self.N_out = len(self.outputs)

        self.num_in = int((args.hist + 1) * self.N_in + self.N_extra)

        print("Number of inputs: ", self.num_in)  # 3 (ocean speeds + ocean temp)(t) +
        # 3 (atm wind stresses + atm temp)(t) +
        # 3 (boundary ocean speeds + boundary ocean temp)(t) -> 3 (ocean speeds + ocean temp)(t+1)
        print("Number of outputs: ", self.N_out)  # 3

        # Post-fix strings
        self.str_train = (
            "steps_"
            + str(args.steps)
            + "_"
            + args.train_region
            + "_Test_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "_out"
            + self.str_out
            + "N_train_4000"
            + "_Lateral_Data_025_no_smooth"
        )
        self.str_save = (
            "steps_"
            + str(args.steps)
            + "_"
            + args.train_region
            + "_"
            + args.region
            + "_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "N_samples_"
            + str(args.N_samples)
        )
        self.post_model_name = (
            "Train_" + args.train_region
            + "_Test_" + args.region
            + "_Test_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "_out"
            + self.str_in
            + "N_train_"
            + str(args.N_samples)
            + "_Lateral_Data_025_no_smooth"
        )
        self.post_pred_name = (
            args.region
            + "_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "N_samples_"
            + str(args.N_samples)
        )

        # Getting start and end indices of train and test
        s_train, e_train, e_test = get_train_test_ranges(
            args.N_samples, args.N_val, args.lag, args.hist, args.interval
        )

        # Saving data
        print("Getting inputs")
        if "global_1" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(self.inputs, self.extra_in, self.outputs, args.lag)
        elif "global_2x" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(self.inputs, self.extra_in, self.outputs, args.lag, run_type ="2x")
        elif "global_4x" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(self.inputs, self.extra_in, self.outputs, args.lag, run_type ="4x")
        else:
            raise NotImplementedError

        print("Calculating mask tensors")
        self.wet, self.wet_nan = get_wet_mask(inputs, "cpu")
        self.wet_bool = np.array(self.wet.cpu()).astype(bool)
        wet_lap = compute_laplacian_wet(self.wet_nan, 4) # hardcoded
        wet_lap = xr.where(wet_lap == 0, 1, np.nan)
        self.wet_lap = np.nan_to_num(wet_lap)
        print("Wet resolution:", self.wet.shape)

        self.time_vec = inputs[0].time.data

        self.time_test = self.time_vec[e_test : (e_test + args.lag * args.N_test)]

        print("Loading Train data")
        train_data = torch.load(
                    Path(args.data_dir) / "train_data_cnn_{0}.pt".format(self.str_train),
                    map_location=torch.device("cpu"),
                )
        # self.train_data = train_data
    
        if args.save_test_data:
            print("Saving data")
            data_in_test = gen_data_in_test(
                0, e_test, args.N_test, args.lag, args.hist, inputs, extra_in
            )
            data_out_test = gen_data_out_test(
                0, e_test, args.N_test, args.lag, args.hist, outputs
            )
            if "global" in args.region:
                norm_vals = train_data.norm_vals
                if "combined" in args.train_region:
                    assert len(norm_vals) == len(GLOBAL_COMBINED_STATS) and all(np.array_equal(norm_vals[k], GLOBAL_COMBINED_STATS[k]) for k in norm_vals)
                self.test_data = data_CNN_Dynamic(
                    data_in_test,
                    data_out_test,
                    self.wet.to(device="cpu"),
                    norm_vals,
                    mode,
                    device=args.device,
                )
                del train_data
            else:
                raise NotImplementedError()
            torch.save(
                self.test_data,
                Path(args.data_dir) / "test_{1}_data_cnn_{0}.pt".format(self.str_save, mode),
            )

        else:
            print("Loading test data")
            self.test_data = torch.load(
                Path(args.data_dir) / "test_{1}_data_cnn_{0}.pt".format(self.str_save, mode)
            )

         # Model
        print("Loading model " + args.network)
        if "swin" in args.network.lower():
            model = instantiate(
                args.swin,
                in_channels=self.num_in,
                output_channels=self.N_in,
                pretrain_img_size=[*self.test_data[0][0].shape[1:]],
                wet=self.wet.cuda()
            )
        elif "unet" in args.network.lower():
            model = instantiate(
                args.unet, wet=self.wet.cuda()
            )

        full_model_path = args.ckpt_path
        self.full_model_name = args.network + "_" + self.post_model_name
        self.output_channels = model.output_channels

        # from torchinfo import summary
        # # summary(model)
        # i = [torch.zeros(1, 9, 128, 192)] * 2
        # summary(model, input_data=[i], col_names=["output_size", "num_params"], depth=4)
        # import pdb; pdb.set_trace()

        model = model.to(args.device)
        model.load_state_dict(
            torch.load(full_model_path, map_location=torch.device(args.device))
        )

        self.model = model

        full_model_path = args.ckpt_path
        self.full_model_name = args.network + "_" + self.post_model_name

        # Stats
        self.mean_out = self.test_data.norm_vals["m_out"]
        self.std_out = self.test_data.norm_vals["s_out"]
        self.mean_in = self.test_data.norm_vals["m_in"]
        self.std_in = self.test_data.norm_vals["s_in"]

        # clim
        self.clim = None
        if args.save_clim_data:
            print("Saving clim")
            clim = np.zeros((366, *self.wet.shape, 3))
            for i in range(self.N_out):
                clim[:, :, :, i] = (
                    outputs[i].groupby("time.dayofyear").mean("time").data
                )
            torch.save(
                clim,
                Path(args.data_dir) / "clim_cnn_{0}.pt".format(self.str_save),
            )

        else:
            print("Loading clim")
            clim = torch.load(
                Path(args.data_dir) / "clim_cnn_{0}.pt".format(self.str_save)
            )

        self.clim = clim

        # Getting area tensor
        print("Computing area tensor")
        self.grids = xr.open_dataset('/scratch/as15415/Data/CM2x_grids/Grid_New.nc').rename({"dx": "dxu", "dy": "dyu"})

        self.area = torch.from_numpy(self.grids["area_C"].to_numpy()).to(device="cpu")
        self.dx = self.grids["dxu"].to_numpy()
        self.dy = self.grids["dyu"].to_numpy()

        # Pred model path dir
        self.pred_model_path = Path(args.path_dir) / self.full_model_name
        if not os.path.isdir(self.pred_model_path):
            os.makedirs(self.pred_model_path)

        self.Nb = args.Nb
        self.hist = args.hist
        self.lag = args.lag
        self.N_test = args.N_test
        self.N_samples = args.N_samples
        self.output_dir = args.output_dir
        self.region = args.region
        self.steps = args.steps
        self.network = args.model_name_replace
        self.inputs = inputs

        self.pred_region = args.region
        self.pred_names = args.pred_names if args.pred_names else []
        self.pred_paths = args.pred_paths if args.pred_paths else []

        self.JUPYTER_MODE = False

    def load_long_data(self):
        print("Load long data...")
        model_pred_net = (
            xr.open_zarr(
                self.pred_model_path
                / (
                    "Pred_lateral_Fast_Data_025_"
                    + self.post_pred_name
                    + "_rand_seed_"
                    + str(1)
                    + ".zarr"
                )
            )
            .to_array()
            .to_numpy()
            .squeeze()
        )

        model_pred_saved_nets = []
        for model_pred_path in self.pred_paths:
            net_path = Path(model_pred_path) / (
                "Pred_lateral_Fast_Data_025_"
                + self.pred_region
                + "_in_"
                + self.str_in
                + "ext_"
                + "tau_u_tau_v_t_ref_"
                + "N_samples_"
                + str(self.N_samples)
                + "_rand_seed_"
                + str(1)
                + ".zarr"
            )

            model_pred_saved_nets.append(
                xr.open_zarr(net_path).to_array().to_numpy().squeeze()
            )
        
        return model_pred_net, model_pred_saved_nets

    def generate_pred_lateral(self):
        print("Generation Pred begin...")
        model_pred = None
        old_pred = None
        for ns in [4000]:
            for rand_ind in range(1, 4):
                print(ns, rand_ind)
                # torch.manual_seed(rand_ind)
                # torch.cuda.manual_seed(rand_ind)
                # import numpy as np
                # np.random.seed(rand_ind)
                # new_test_data = copy.deepcopy(self.test_data)
                old_pred = model_pred
                model_pred = generate_model_rollout(
                    self.N_test,
                    self.test_data,
                    self.model,
                    self.hist,
                    self.N_in,
                    self.N_extra,
                    self.Nb,
                    self.region,
                )

                print("data_gen")
                da = xr.DataArray(
                    data=model_pred,
                    dims=["time", "x", "y", "var"],
                )

                da.to_zarr(
                    self.pred_model_path
                    / (
                        "Pred_lateral_Fast_Data_025_"
                        + self.post_pred_name
                        + "_rand_seed_"
                        + str(rand_ind)
                        + ".zarr"
                    ),
                    mode="w",
                )
                print(f"Model pred shape {model_pred.shape}")
                # del model_pred
        print((old_pred == model_pred).all())

    def send_data_to_cpu(self):
        self.test_data.set_device(device="cpu")

In [26]:
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
import copy
from datetime import datetime
import os

# G1, G1
with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
    args1 = compose(config_name="exp/eval_unet_global", overrides=[
        "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
        "model_name_replace=ConvNext UNet",
        "network=ConvNext UNet Train1Eval1-default",
        "train_region=global_1",
        "region=global_1",
        "save_test_data=False", # Done Generation
        "run_gen_pred=False", # Done Generation
        "ckpt_path=/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt",
        "pred_names=null",
        "pred_paths=null"
    ])
if not os.path.exists(args1.output_dir):
    os.mkdir(args1.output_dir)

e1 = Eval(args1, mode='default')
if args1.run_gen_pred:
    e1.generate_pred_lateral()
else:
    print("Skipping pred generation")
e1.send_data_to_cpu()

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs
Calculating mask tensors


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model ConvNext UNet Train1Eval1-default
Loading clim
Computing area tensor
Generation Pred begin...
4000 1
[[-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
4000 2
[[-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
4000 3
[[-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
True


In [27]:
# G1, G1
with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
    args2 = compose(config_name="exp/eval_unet_global", overrides=[
        "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
        "model_name_replace=ConvNext UNet |-1|",
        "network=ConvNext UNet Train1Eval1-minus",
        "save_test_data=False", # Done Generation
        "run_gen_pred=False", # Done Generation
        "train_region=global_1",
        "region=global_1",
        "ckpt_path=/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt",
        "pred_names=null",
        "pred_paths=null"
    ])
if not os.path.exists(args2.output_dir):
    os.mkdir(args2.output_dir)

e2 = Eval(args2, mode='minus')
if args2.run_gen_pred:
    e2.generate_pred_lateral()
else:
    print("Skipping pred generation")
e2.send_data_to_cpu()

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs
Calculating mask tensors


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model ConvNext UNet Train1Eval1-minus
Loading clim
Computing area tensor
Generation Pred begin...
4000 1
[[-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
4000 2
[[-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
4000 3
[[-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
True


In [28]:
# G1, G1
with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
    args3 = compose(config_name="exp/eval_unet_global", overrides=[
        "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
        "model_name_replace=ConvNext UNet |+1|",
        "network=ConvNext UNet Train1Eval1-plus",
        "save_test_data=False", # Done Generation
        "run_gen_pred=False", # Done Generation
        "train_region=global_1",
        "region=global_1",
        "ckpt_path=/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt",
        "pred_names=null",
        "pred_paths=null"
    ])
if not os.path.exists(args3.output_dir):
    os.mkdir(args3.output_dir)

e3 = Eval(args3, mode='plus')
if args3.run_gen_pred:
    e3.generate_pred_lateral()
else:
    print("Skipping pred generation")
e3.send_data_to_cpu()

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs
Calculating mask tensors


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model ConvNext UNet Train1Eval1-plus
Loading clim
Computing area tensor
Generation Pred begin...
4000 1
[[-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
4000 2
[[-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
4000 3
[[-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
True


In [41]:
def get_timeseries_temperature(e, model_pred_net, model_pred_saved_nets):
    ### Spatial matching metrics
    print("Getting Spatial matching stats...")
    T_test = np.array(
        e.test_data[:][1][:, 2] * e.std_out[2] + e.mean_out[2]
    )

    print("Getting Mean...")
    mean_T_net, mean_T_true = compute_mean_single(
        3000,
        T_test,
        model_pred_net[:, :, :, 2],
        e.area,
        e.wet_bool,
    )
    mean_T_saved = []
    for model_pred_saved in model_pred_saved_nets:
        mean_T_i, mean_T_true = compute_mean_single(
            3000,
            T_test,
            model_pred_saved[:, :, :, 2],
            e.area,
            e.wet_bool,
        )
        mean_T_saved.append(mean_T_i)

    return mean_T_true, mean_T_saved + [mean_T_net]

In [42]:

model_pred_net, model_pred_saved_nets = e1.load_long_data()
mean_T_true1, mean_T_saved1 = get_timeseries_temperature(e1, model_pred_net, model_pred_saved_nets)
model_pred_net, model_pred_saved_nets = e2.load_long_data()
mean_T_true2, mean_T_saved2 = get_timeseries_temperature(e2, model_pred_net, model_pred_saved_nets)
model_pred_net, model_pred_saved_nets = e3.load_long_data()
mean_T_true3, mean_T_saved3 = get_timeseries_temperature(e3, model_pred_net, model_pred_saved_nets)


Load long data...
Getting Spatial matching stats...
Getting Mean...
Load long data...
Getting Spatial matching stats...
Getting Mean...
Load long data...
Getting Spatial matching stats...
Getting Mean...


In [70]:
def plot_metrics_mean(
    network_names,
    region,
    save_str,
    output_dir,
    mean_T_true1,
    mean_Ts1,
    mean_T_true2,
    mean_Ts2,
    mean_T_true3,
    mean_Ts3,
    start=1999,
    end=2999,
    JUPYTER_MODE=False,
):
    plt.style.use("bmh")

    clist = ["#A00B41", "#3300EA", "#00DCDE", "#A6BD00"]


    # Temp 1
    N_eval = len(mean_T_true1)
    plt.plot(
        np.arange(start, end),
        mean_Ts1[0][start:end],
        c=clist[0],
        label=r"0$^\circ C$ Perturbation"
    )

    # Temp 2
    N_eval = len(mean_T_true2)
    plt.plot(
        np.arange(start, end),
        mean_Ts2[0][start:end],
        c=clist[1],
        label=r"-1$^\circ C$ Perturbation"
    )


    # Temp 3
    N_eval = len(mean_T_true3)
    plt.plot(
        np.arange(start, end),
        mean_Ts3[0][start:end],
        c=clist[2],
        label=r"+1$^\circ C$ Perturbation"
    )

    plt.plot(np.arange(start, end), mean_T_true1[start:end], "--k", label="CM2.6")
    plt.xlabel(r"time $( days )$")
    plt.ylabel(r"$\overline{T}$ $( ^\circ C )$")
    # plt.ylim([0, 1])
    plt.xlim([start, end])
    plt.title(r"Perturbation Plots")

    plt.legend(bbox_to_anchor=(0, 1.2, 1, 0.2), loc="lower left", fancybox=True, ncol=len(mean_Ts1)+1)





    
    # plt.show()
    
    plt.savefig(
        Path(output_dir) / ("Mean" + region + "_" + save_str + ".png"),
        bbox_inches="tight",
    )
    plt.clf()

In [71]:
plot_metrics_mean(
    e1.pred_names + [e1.network],
    e1.region + '_Long_',
    e1.str_save,
    e1.output_dir,
    mean_T_true1, mean_T_saved1,
    mean_T_true2, mean_T_saved2,
    mean_T_true3, mean_T_saved3
    )

<Figure size 640x480 with 0 Axes>